In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, roc_auc_score

from xgboost import XGBClassifier

df = pd.read_csv("SampleTransaction12.csv", header=None)
df = df[0].str.split(",", expand=True)

# Set header
df.columns = df.iloc[0]
df = df[1:]

df.head()

# Convert amount
df['amount'] = df['amount'].str.replace('$', '').astype(float)

# Convert date
df['date'] = pd.to_datetime(df['date'])

# Drop useless column
df = df.drop(columns=['None'], errors='ignore')

df.info()

import json

with open("train_fraud_labels.json") as f:
    labels = json.load(f)

labels_df = pd.DataFrame(list(labels['target'].items()), columns=['id', 'fraud'])

df['id'] = df['id'].astype(str)
labels_df['id'] = labels_df['id'].astype(str)

df = df.merge(labels_df, on='id', how='left')

# Time features
df['hour'] = df['date'].dt.hour
df['day'] = df['date'].dt.dayofweek

# High amount flag
df['high_amount'] = (df['amount'] > df['amount'].quantile(0.95)).astype(int)

# Error flag
df['has_error'] = df['errors'].notna().astype(int)

# MCC risk (manual)
high_risk_mcc = ['4829', '7995']
df['high_risk_mcc'] = df['mcc'].isin(high_risk_mcc).astype(int)

le = LabelEncoder()

df['merchant_city'] = le.fit_transform(df['merchant_city'].astype(str))
df['merchant_state'] = le.fit_transform(df['merchant_state'].astype(str))
df['use_chip'] = le.fit_transform(df['use_chip'].astype(str))

df['fraud'] = df['fraud'].map({'Yes':1, 'No':0})

df = df.dropna(subset=['fraud'])

features = [
    'amount', 'hour', 'day',
    'high_amount', 'has_error',
    'high_risk_mcc',
    'merchant_city', 'merchant_state', 'use_chip'
]

X = df[features]
y = df['fraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = XGBClassifier(scale_pos_weight=10)  # handle imbalance
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_pred))

import joblib

joblib.dump(model, "fraud_model.pkl")

